<a href="https://colab.research.google.com/github/shaheem1771/Movie-Recommendation-System/blob/main/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-06-16 16:00:32--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  --.-KB/s    in 0.1s    

2026-06-16 16:00:33 (6.22 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [ ]:
import pandas as pd

movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies['features'] = movies['title'] + " " + movies['genres']

tfidf = TfidfVectorizer(stop_words='english')

matrix = tfidf.fit_transform(movies['features'])

cosine_sim = cosine_similarity(matrix)

In [ ]:
tags = pd.read_csv("ml-latest-small/tags.csv")

tag_data = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

movies = movies.merge(tag_data, on='movieId', how='left')

movies['tag'] = movies['tag'].fillna('')

movies['features'] = (
    movies['title']
    + ' '
    + movies['genres']
    + ' '
    + movies['tag']
)

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['features'])

cosine_sim = cosine_similarity(tfidf_matrix)

print(tfidf_matrix.shape)
print(cosine_sim.shape)

(9742, 9949)
(9742, 9742)


In [ ]:
recommend("Toy Story")
recommend("Batman")
recommend("Avengers")

['Masked Avengers (1981)',
 'Ultimate Avengers 2 (2006)',
 'Avengers: Age of Ultron (2015)',
 'Ultimate Avengers (2006)',
 'Whatever (1998)',
 'Crippled Avengers (Can que) (Return of the 5 Deadly Venoms) (1981)',
 'Lost in Space (1998)',
 'Legionnaire (1998)',
 'Merlin (1998)',
 'Civil Action, A (1998)']

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

(9742, 9742)


In [ ]:
def recommend(movie_name, num_recommendations=10):

    movie_name = movie_name.lower()

    matching_movies = movies[
        movies['title'].str.lower().str.contains(movie_name)
    ]

    if matching_movies.empty:
        return ["Movie not found"]

    idx = matching_movies.index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in similarity_scores[1:num_recommendations+1]:
        recommendations.append(movies.iloc[i]['title'])

    return recommendations

In [ ]:
!pip install fuzzywuzzy python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.6 MB/s eta 0:00:00


In [ ]:
from fuzzywuzzy import process

movie_titles = movies['title'].tolist()

def find_movie(movie_name):
    match = process.extractOne(movie_name, movie_titles)
    return match[0]

In [ ]:
print(find_movie("Batman"))
print(find_movie("Avengers"))
print(find_movie("Spider Man"))
print(find_movie("Harry Potter"))

Batman Forever (1995)
Avengers, The (1998)
Spider-Man (2002)
Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)


In [ ]:
from fuzzywuzzy import process

movie_titles = movies['title'].tolist()

def recommend(movie_name, num_recommendations=10):

    # Find closest movie title
    closest_match = process.extractOne(movie_name, movie_titles)[0]

    # Find index
    idx = movies[movies['title'] == closest_match].index[0]

    # Similarity scores
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    # Skip first result (same movie)
    for i, score in similarity_scores[1:num_recommendations+1]:
        recommendations.append(movies.iloc[i]['title'])

    return recommendations

In [ ]:
recommend("Batman")

['Batman (1989)',
 'Batman Returns (1992)',
 'Superhero Movie (2008)',
 'Mystery Men (1999)',
 'X-Men (2000)',
 'Superman (1978)',
 'Spider-Man (2002)',
 'Supergirl (1984)',
 'Superman III (1983)',
 'Superman II (1980)']

In [ ]:
recommend("Barbie")

['Barbershop 2: Back in Business (2004)',
 'Barbershop: The Next Cut (2016)',
 'All or Nothing (2002)',
 'Enough (2002)',
 'Below (2002)',
 'They (2002)',
 'Friday After Next (2002)',
 'Trip, The (2002)',
 'In This World (2002)',
 'Notorious C.H.O. (2002)']

In [ ]:
recommend("Harry Potter")

['Harry Potter and the Goblet of Fire (2005)',
 'Harry Potter and the Chamber of Secrets (2002)',
 'Harry Potter and the Prisoner of Azkaban (2004)',
 'Harry Potter and the Order of the Phoenix (2007)',
 'Harry Potter and the Deathly Hallows: Part 1 (2010)',
 'Harry Potter and the Deathly Hallows: Part 2 (2011)',
 'Harry Potter and the Half-Blood Prince (2009)',
 'Miss Potter (2006)',
 'Very Potter Musical, A (2009)',
 'Very Potter Sequel, A (2010)']

In [ ]:
recommend("lord of the rings")

['Lord of the Rings: The Two Towers, The (2002)',
 'Lord of the Rings: The Return of the King, The (2003)',
 'Lord of the Rings: The Fellowship of the Ring, The (2001)',
 'Wiz, The (1978)',
 'Watership Down (1978)',
 'Lord of the Flies (1990)',
 'Wild, The (2006)',
 'Three from Prostokvashino (1978)',
 'Lord of Illusions (1995)',
 'Dragon ball Z 04: Lord Slug (1991)']

In [ ]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

user_movie_matrix_filled.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(metric='cosine', algorithm='brute')

model.fit(user_movie_matrix_filled)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
distances, indices = model.kneighbors(
    user_movie_matrix_filled.iloc[0, :].values.reshape(1, -1),
    n_neighbors=6
)

print(indices)
print(distances)

[[  0 265 312 367  56  90]]
[[2.22044605e-16 6.42592290e-01 6.48438482e-01 6.54872948e-01
  6.54965721e-01 6.65273080e-01]]


In [ ]:
for i in range(len(distances.flatten())):
    if i == 0:
        print("Recommendations for User 1:\n")
    else:
        print(
            f"User {indices.flatten()[i]} "
            f"with distance {distances.flatten()[i]}"
        )

Recommendations for User 1:

User 265 with distance 0.6425922903967258
User 312 with distance 0.6484384815090432
User 367 with distance 0.6548729484164695
User 56 with distance 0.6549657211924163
User 90 with distance 0.6652730795818291


In [ ]:
!pip install fuzzywuzzy python-Levenshtein

In [ ]:
from fuzzywuzzy import process

In [ ]:
def find_movie(title):
    titles = movies['title'].tolist()

    closest_match = process.extractOne(title, titles)

    return closest_match[0]

In [ ]:
print(find_movie("batman"))
print(find_movie("harry potter"))
print(find_movie("avengers"))
print(find_movie("barbie"))

Batman Forever (1995)
Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Avengers, The (1998)
Barbershop (2002)


In [ ]:
def find_movie(title):
    titles = movies['title'].tolist()

    match, score = process.extractOne(title, titles)

    if score < 80:
        return "Movie not found"

    return match

In [ ]:
print(find_movie("batman"))
print(find_movie("harry potter"))
print(find_movie("avengers"))
print(find_movie("barbie"))
print(find_movie("oppenheimer"))

Batman Forever (1995)
Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Avengers, The (1998)
Movie not found
Movie not found


In [ ]:
def hybrid_recommend(movie_name):
    corrected_movie = find_movie(movie_name)

    if corrected_movie == "Movie not found":
        return ["Movie not found"]

    return recommend(corrected_movie)

In [ ]:
hybrid_recommend("Batman")

['Batman (1989)',
 'Batman Returns (1992)',
 'Superhero Movie (2008)',
 'Mystery Men (1999)',
 'X-Men (2000)',
 'Superman (1978)',
 'Spider-Man (2002)',
 'Supergirl (1984)',
 'Superman III (1983)',
 'Superman II (1980)']

In [ ]:
hybrid_recommend("Toy Story")

['Toy Story 2 (1999)',
 "Bug's Life, A (1998)",
 'Toy Story 3 (2010)',
 'Toy, The (1982)',
 'Up (2009)',
 'Fun (1994)',
 "We're Back! A Dinosaur's Story (1993)",
 'Now and Then (1995)',
 'Toy Soldiers (1991)',
 'NeverEnding Story, The (1984)']

In [ ]:
def recommend_with_genres(movie_name, num_recommendations=10):
    recommendations = hybrid_recommend(movie_name)

    if recommendations == ["Movie not found"]:
        return

    print(f"\nRecommendations for {movie_name}:\n")

    for movie in recommendations[:num_recommendations]:
        genre = movies[movies['title'] == movie]['genres'].values[0]
        print(f"{movie}  |  {genre}")

In [ ]:
recommend_with_genres("Toy Story")


Recommendations for Toy Story:

Toy Story 2 (1999)  |  Adventure|Animation|Children|Comedy|Fantasy
Bug's Life, A (1998)  |  Adventure|Animation|Children|Comedy
Toy Story 3 (2010)  |  Adventure|Animation|Children|Comedy|Fantasy|IMAX
Toy, The (1982)  |  Comedy
Up (2009)  |  Adventure|Animation|Children|Drama
Fun (1994)  |  Crime|Drama
We're Back! A Dinosaur's Story (1993)  |  Adventure|Animation|Children|Fantasy
Now and Then (1995)  |  Children|Drama
Toy Soldiers (1991)  |  Action|Drama
NeverEnding Story, The (1984)  |  Adventure|Children|Fantasy


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
def movie_app(movie_name):

    recommendations = hybrid_recommend(movie_name)

    if recommendations == ["Movie not found"]:
        return "Movie not found"

    output = []

    for movie in recommendations:

        genre = movies[
            movies['title'] == movie
        ]['genres'].values[0]

        output.append(
            f"🎬 {movie}\nGenre: {genre}"
        )

    return "\n\n".join(output)

In [ ]:
demo = gr.Interface(
    fn=movie_app,
    inputs=gr.Textbox(
        label="Movie Name",
        placeholder="Enter a movie..."
    ),
    outputs=gr.Textbox(
        label="Recommendations",
        lines=12
    ),
    title="🎬 Movie Recommendation System",
    description="Get movie recommendations based on similar movies."
)

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7efdf8cd02af04fa7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [6]:
!git config --global user.name "shaheem1771"
!git config --global user.email "shaheem1771@gmail.com"

In [7]:
!git remote add origin https://github.com/shaheem1771/Movie-Recommendation-System.git

In [8]:
!git add .
!git commit -m "Initial commit"
!git branch -M main

[master (root-commit) 926e9f4] Initial commit
 21 files changed, 51059 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.06.04/13.31.42.499627.log
 create mode 100644 .config/logs/2026.06.04/13.32.02.654775.log
 create mode 100644 .config/logs/2026.06.04/13.32.18.735754.log
 create mode 100644 .config/logs/2026.06.04/13.32.21.210570.log
 create mode 100644 .config/logs/2026.06.04/13.32.38.346437.log
 create mode 100644 .config/logs/2026.06.04/13.32.39.344962.log
 create mode 100755 sample_data/README.md
 create mode 1007

In [9]:
!git push -u origin main

fatal: could not read Username for 'https://github.com': No such device or address
